# ColorCorrect: compare fit methods and NN options

Audience:
- Researchers comparing calibration models, with basic Python knowledge.

Prerequisites:
- Python 3.10+, Jupyter and Pillow for thumbnails (`pip install jupyterlab pillow`), built ColorCorrect worker and example corpus. Set COLORCORRECT_SAMPLES if needed. No Python machine-learning packages required.

Learning goals:
- Compare chart-fit versus held-out DeltaE00, time the same workload, and configure NN layers, optimizer, batch normalization and dropout. This small benchmark is illustrative, not a universal model ranking.


## Outline

1. Setup
2. Detect once, then compare all six fit families
3. Compare three NN optimizers with identical topology
4. Interpret timings and generalization


In [ ]:
import os, sys, tempfile, json
from pathlib import Path
from time import perf_counter
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'api/colorcorrect.py').is_file()), None)
assert root, 'Run from the app folder or notebooks/ subfolder'
sys.path.insert(0, str(root / 'api'))
from colorcorrect import Engine
samples = Path(os.environ.get('COLORCORRECT_SAMPLES', str(root / 'test_images'))).resolve()
chart, white = samples / 'ColorCard_Big.JPG', samples / 'Blank.JPG'
assert chart.is_file() and white.is_file()
out = Path(tempfile.mkdtemp(prefix='colorcorrect-models-'))
rows = []
print('Artifacts:', out)


## Compare fit families

Keep FFC/GC/WB and the chart fixed. Each timed request includes image load, fitting, held-out validation and report/model serialization but excludes detection and final-image encoding (write_output=False). First-run caches affect timing; repeat for serious benchmarking. NN uses 200 epochs for tutorial speed, not a recommended production optimum.


In [ ]:
with Engine(root=root, policy='cpu') as engine:
    detection = engine.detect(chart, handshake_out=out / 'chart.json')
    for method in ['ccm', 'rpcc2', 'rpcc3', 'linear', 'pls', 'nn']:
        t = perf_counter()
        result = engine.fit(chart, handshake=detection['handshake'], output=out / f'{method}.png',
            report=out / f'{method}.report.json', model_out=out / f'{method}.model.json',
            model=method, degree=2, white=white, do_ffc=True, do_gc=True, do_wb=True,
            write_output=False, phase2_teacher=False, cross_validate=True,
            nn={'hidden_layers': [64, 32], 'optimizer': 'adam', 'epochs': 200, 'patience': 50})
        report = result['report_json']
        assert report['do_ffc'], 'FFC unexpectedly skipped'
        row = {'method': method, 'seconds': round(perf_counter() - t, 3),
               'training': report['delta_e_2000']['mean_after'],
               'validation': report.get('validation_delta_e_2000', {})}
        rows.append(row)
        print(json.dumps(row))


## NN options

The API accepts an integer (one hidden layer) or array of integers. The native model supports one to four layers, widths 4–512; dropout 0–0.8. Optimizers are adam, sgd (momentum), and rmsprop. Compare optimizers with identical layers, normalization, dropout and epoch budget.


In [ ]:
with Engine(root=root, policy='cpu') as engine:
    for optimizer in ['adam', 'sgd', 'rmsprop']:
        result = engine.fit(chart, handshake=detection['handshake'], output=out / f'{optimizer}.png',
            report=out / f'{optimizer}.report.json', model_out=out / f'{optimizer}.model.json',
            model='nn', do_ffc=True, white=white, write_output=False, phase2_teacher=False,
            nn={'hidden_layers': [64, 32], 'optimizer': optimizer, 'batch_norm': True,
                'dropout': 0.1, 'epochs': 200, 'patience': 50, 'lr': 0.001})
        report = result['report_json']
        print(optimizer, 'train:', round(report['delta_e_2000']['mean_after'], 3),
              'validation:', report.get('validation_delta_e_2000'))


## Visual sanity check
Preview the final RMSprop example, with the same white reference and saved FFC settings. This illustrates how to inspect a fitted model, not a claim that RMSprop is universally best. The original and corrected images below come from this run.


In [ ]:
with Engine(root=root, policy='cpu') as engine:
    engine.preview(out / 'rmsprop.model.json', chart, 'cc', out / 'nn_preview', do_ffc=True, white=white)
from io import BytesIO
from PIL import Image as PillowImage
from IPython.display import Image, Markdown, display
for path, caption in [(chart, 'Original'), (out / 'nn_preview/corrected.png', 'NN RMSprop example after FFC → GC → WB → CC')]:
    with PillowImage.open(path) as original:
        thumb = original.convert('RGB')
        thumb.thumbnail((640, 420))
        buffer = BytesIO()
        thumb.save(buffer, format='JPEG', quality=88)
    display(Markdown('**' + caption + '**'), Image(data=buffer.getvalue()))


## Exercise: choose a model honestly
Do not choose only the lowest training DeltaE. Inspect held-out validation and fit warnings, repeat timings, and validate a separate capture. As an extension, set chart to ColorCard_Combo.JPG and detect again; never reuse another image's handshake. The spatial flat field must match the capture geometry.

The answer scaffold below only ranks training fits to illustrate why this ranking is insufficient. Read the validation dictionaries above before making a decision. See the [API guide](../docs/api/README.md) and [original paper](https://acsess.onlinelibrary.wiley.com/doi/full/10.1002/ppj2.70067).


In [ ]:
print('Training-only ranking (not a recommendation):')
for row in sorted(rows, key=lambda r: r['training']):
    print(row['method'], round(row['training'], 3), 'seconds:', row['seconds'])
